# PCR-GLOB experiments

example of how to run PCR-GLOBWB

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from cartopy import feature as cfeature
from matplotlib.colors import LogNorm
from rich import print
from tqdm.notebook import tqdm

import ewatercycle.forcing
import ewatercycle.models
import ewatercycle.observation.grdc
import ewatercycle.parameter_sets
from ewatercycle.container import ContainerImage

PROJECT_ROOT = Path().resolve().parents[2]
sys.path.append(str(PROJECT_ROOT))

from src.aral import load_grdc_monthly
from src.constants import STATIONS_PCR
from src.paths import FORCING_PCRGLOB, GRDC, INI_FILES, LOAD_PCR, PCR_GLOBAL_PARAMS, PCR_TAIL

/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


In [2]:
STATIONS_PCR

{'Chatly': {'lat': 42.2, 'lon': 60.2},
 'Kazalinsk': {'lat': 45.7, 'lon': 62.12},
 'Kerki': {'lat': 37.83, 'lon': 65.3},
 'Tyumen-Aryk': {'lat': 43.95, 'lon': 67.05},
 'Uch-Kurgan': {'lat': 41.12, 'lon': 72.1},
 'Garm': {'lat': 39.05, 'lon': 70.33}}

In [3]:
pcr_glob_directory = PCR_GLOBAL_PARAMS
prepared_pcr_forcing = FORCING_PCRGLOB / LOAD_PCR / PCR_TAIL

parameter_set_test = ewatercycle.parameter_sets.ParameterSet(
    name="custom_parameter_set",
    directory=pcr_glob_directory,
    config=INI_FILES / "calibrated_best_save.ini",
    target_model="pcrglobwb",
    supported_model_versions={"aral"},
)

forcing = ewatercycle.forcing.sources["PCRGlobWBForcing"].load(
    directory=prepared_pcr_forcing,
)

In [4]:
my_image = ContainerImage("/home/avandervee3/pcrglob_aral.sif")
my_image.version

'aral'

In [5]:
reference = ewatercycle.models.PCRGlobWB(
    parameter_set=parameter_set_test, forcing=forcing, bmi_image=my_image
)

reference_2x = ewatercycle.models.PCRGlobWB(
    parameter_set=parameter_set_test, forcing=forcing, bmi_image=my_image
)

reference_multiple = ewatercycle.models.PCRGlobWB(
    parameter_set=parameter_set_test, forcing=forcing, bmi_image=my_image
)   


print(reference)

PCRGlobWB(
    parameter_set=ParameterSet(
        name='custom_parameter_set',
        directory=PosixPath('/data/shared/parameter-sets/pcrglobwb_global'),
        config=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_i
n_progress/model_runs/pcrglobwb/ini_files/calibrated_best_save.ini'),
        doi='N/A',
        target_model='pcrglobwb',
        supported_model_versions={'aral'},
        downloader=None
    ),
    forcing=PCRGlobWBForcing(
        start_time='1940-01-01T00:00:00Z',
        end_time='1960-12-31T00:00:00Z',
        directory=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/wor
k_in_progress/forcing/output/PCRGLOBWB/ERA5_1940-1960/AralSea_basin/work/diagnostic/script'),
        shape=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in
_progress/forcing/output/PCRGLOBWB/ERA5_1940-1960/AralSea_basin/work/diagnostic/script/AralSea_basin.shp'),
        filenames={},
        precipitationNC='pcrglobwb_OBS6_ERA5_reanaly_1_day_pr_1940-1960_AralSea_basin.nc',
        temperatureNC='pcrglobwb_OBS6_ERA5_reanaly_1_day_tas_1940-1960_AralSea_basin.nc'
    )
)

In [6]:
experiment_start_date = "1950-06-01T00:00:00Z"
experiment_end_date = "1950-07-30T00:00:00Z"

In [7]:
from pathlib import Path
import shutil

for cfg_dir in [Path("channel_experiment_reference"), Path("channel_experiment_set_storage"), Path("channel_experiment_set_multiple_storage")]:
    if cfg_dir.exists():
        shutil.rmtree(cfg_dir)

reference_config, reference_dir = reference.setup(
    start_time=experiment_start_date, end_time=experiment_end_date, max_spinups_in_years=0, cfg_dir="channel_experiment_reference"
)
reference_config, reference_dir

reference_2x_config, reference_2x_dir = reference_2x.setup(
    start_time=experiment_start_date, end_time=experiment_end_date, max_spinups_in_years=0, cfg_dir="channel_experiment_set_storage"
)
reference_2x_config, reference_2x_dir

reference_multiple_config, reference_multiple_dir = reference_multiple.setup(
    start_time=experiment_start_date, end_time=experiment_end_date, max_spinups_in_years=0, cfg_dir="channel_experiment_set_multiple_storage"
)
reference_multiple_config, reference_multiple_dir

('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/Report/notebooks/channel_experiment_set_multiple_storage/pcrglobwb_ewatercycle.ini',
 '/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/Report/notebooks/channel_experiment_set_multiple_storage')

In [8]:
print(reference.parameters)

refence_para = reference.parameters

# Convert ISO 8601 strings to datetime objects
start_time = datetime.strptime(experiment_start_date, "%Y-%m-%dT%H:%M:%SZ")
end_time = datetime.strptime(experiment_end_date, "%Y-%m-%dT%H:%M:%SZ")

# Calculate the number of days for the progression bar
delta = end_time - start_time
number_of_days = delta.days
print(f"Number of days to model: {number_of_days}")

dict_items([('start_time', '1950-06-01T00:00:00Z'), ('end_time', '1950-07-30T00:00:00Z'), ('routing_method', 
'accuTravelTime'), ('max_spinups_in_years', '0')])

Number of days to model: 59

In [9]:
reference.initialize(reference_config)
reference_2x.initialize(reference_2x_config)
reference_multiple.initialize(reference_multiple_config)

In [10]:
time = pd.date_range(reference.start_time_as_isostr, reference.end_time_as_isostr)
stations_timeseries = pd.DataFrame(
    index=pd.Index(time, name="time"), columns=["Chatly", "Kerki", "Tyumen", "Kazalinsk"]
)
stations_timeseries.head()

,Chatly,Kerki,Tyumen,Kazalinsk
time,,,,
1950-06-01 00:00:00+00:00,NaN,NaN,NaN,NaN
1950-06-02 00:00:00+00:00,NaN,NaN,NaN,NaN
1950-06-03 00:00:00+00:00,NaN,NaN,NaN,NaN
1950-06-04 00:00:00+00:00,NaN,NaN,NaN,NaN
1950-06-05 00:00:00+00:00,NaN,NaN,NaN,NaN


In [11]:
lat_idx = 61
lon_idx = 127

coords_3 = [
    (61, 127),
    (61, 128),
    (60, 128),
    (59, 129),
    (59, 130),
    (58, 131),
    (62, 126),
    (62, 125),
    (63, 124),
    (63, 123),
    (63, 122),
    (63, 121)
]


In [ ]:
# for i in tqdm(range(number_of_days), desc="Running model"):
#     reference.update()
#     # reference_2x.update()
#     # reference_multiple.update()
# print("Model run finished!")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

# Define your fixed removal target amount
#DAILY_REMOVAL_TARGET = 10000.0  
DAILY_REMOVAL_TARGET = 10 * 1e9 / 365   #10  cubic kilometers per year converted to cubic meters per day


history_2x = []
history_multiple = []
timestamps = []

for i in tqdm(range(number_of_days), desc="Running model"):
    reference.update()
    reference_2x.update()
    reference_multiple.update()

    current_time = pd.to_datetime(reference.time_as_isostr)
    timestamps.append(current_time)

    # --- 1. Perturb reference_2x (Single cell) ---
    test_variable = reference_2x.get_value_as_xarray("channel_storage")
    test_values = test_variable.values.copy()
    
    old_value = test_values[lat_idx, lon_idx]
    
    if old_value > 0:
        # Calculate factor to leave behind (Old - Target) / Old
        calculated_factor = (old_value - DAILY_REMOVAL_TARGET) / old_value
    else:
        calculated_factor = 1.0
        
    # Bound the factor between 0.2 (max 80% removal) and 1.0 (0% removal)
    dynamic_factor_2x = np.clip(calculated_factor, 0.2, 1.0)
    
    # Apply and track exactly how much was actually taken
    test_values[lat_idx, lon_idx] *= dynamic_factor_2x
    amount_changed_2x = test_values[lat_idx, lon_idx] - old_value
    
    history_2x.append(amount_changed_2x)
    reference_2x.set_value("channel_storage", test_values.flatten())


    # --- 2. Perturb reference_multiple (Multiple cells) ---
    test_variable_multiple = reference_multiple.get_value_as_xarray("channel_storage")
    test_values_multiple = test_variable_multiple.values.copy()
    
    # For multiple cells, let's divide the 10,000 target equally among all active points
    # (Alternatively, you could pull 10,000 from EACH cell by omitting the division)
    target_per_cell = DAILY_REMOVAL_TARGET / len(coords_3)
    
    timestep_multiple_changes = {}
    for l_idx, m_idx in coords_3:
        old_val_m = test_values_multiple[l_idx, m_idx]
        
        if old_val_m > 0:
            calculated_m_factor = (old_val_m - target_per_cell) / old_val_m
        else:
            calculated_m_factor = 1.0
            
        dynamic_factor_m = np.clip(calculated_m_factor, 0.2, 1.0)
        
        test_values_multiple[l_idx, m_idx] *= dynamic_factor_m
        diff_m = test_values_multiple[l_idx, m_idx] - old_val_m
        
        timestep_multiple_changes[(l_idx, m_idx)] = diff_m
        
    history_multiple.append(timestep_multiple_changes)
    reference_multiple.set_value("channel_storage", test_values_multiple.flatten())

print("Model run finished!")

Running model:  22%|██▏       | 13/59 [02:43<08:15, 10.78s/it]